Some prices didn't get saved in the db, likely an issue with parsing string

-- scrap this, the prices were there, it's the sql script that broke them, so re-run than instead of doing this jerry rigging --

In [38]:
import pandas as pd
import json

# Read the JSON file
with open('./scrapes_dump.json', 'r') as f:
    data_str = f.read()

# Convert string to list by wrapping in brackets
if data_str.rstrip().endswith(','):
    data_str = data_str.rstrip().rstrip(',')
if not data_str.startswith('['):
    data_str = '[' + data_str + ']'

data = json.loads(data_str)

# Convert to DataFrame
json_scrapes_df = pd.DataFrame(data)

# Convert createdAt to datetime
json_scrapes_df['createdAt'] = pd.to_datetime(json_scrapes_df['createdAt'])

# Optional: Sort by reference_item_id and createdAt
json_scrapes_df = json_scrapes_df.sort_values(['createdAt'], ascending=False)

json_scrapes_df

,name,quantity,unitOfMeasure,price,pricePerWeight,referenceUrl,createdAt,reference_item_id
321,Dry Black Chickpeas,1.81,kg,3.99,2.204420,https://www.foodbasics.ca/aisles/pantry/canned...,2025-02-02 03:34:21.505000+00:00,2
320,Black Eyed Peas,907.0,g,3.49,0.003848,https://www.foodbasics.ca/aisles/pantry/pasta-...,2025-02-02 03:34:21.505000+00:00,2
319,Canned Refried Black Beans,454.0,g,3.49,0.007687,https://www.foodbasics.ca/aisles/pantry/canned...,2025-02-02 03:34:21.504000+00:00,2
318,Black-Eyed Peas,900.0,g,2.99,0.003322,https://www.foodbasics.ca/aisles/pantry/canned...,2025-02-02 03:34:21.503000+00:00,2
317,Black Turtle Beans,907.0,g,2.99,0.003297,https://www.foodbasics.ca/aisles/pantry/pasta-...,2025-02-02 03:34:21.502000+00:00,2
...,...,...,...,...,...,...,...,...
4,Chicken Bouillon Cubes,69,g,2.79,0.040435,https://www.foodbasics.ca/aisles/pantry/herbs-...,2025-01-27 03:04:40.077000+00:00,3
3,Halal Chicken Flavoured Bouillon Cubes,80,g,1.29,0.016125,https://www.foodbasics.ca/aisles/pantry/herbs-...,2025-01-27 03:04:40.077000+00:00,3
2,Garlic BBQ Sauce,455,mL,3.29,0.007231,https://www.foodbasics.ca/aisles/pantry/condim...,2025-01-27 03:04:36.800000+00:00,1
1,BBQ Sauce,455,mL,3.29,0.007231,https://www.foodbasics.ca/aisles/pantry/condim...,2025-01-27 03:04:36.799000+00:00,1


In [39]:
# Set pandas display options to show max 10 rows
pd.set_option('display.max_rows', 10)


In [ ]:
import sqlite3

# Connect to SQLite database
conn = sqlite3.connect('../receipts-app/prisma/dev.db')

# Query ProductPriceProof table into DataFrame
df_proofs = pd.read_sql_query("SELECT * FROM ProductPriceProof", conn)

# Close connection
conn.close()

df_proofs

,id,name,quantity,unitOfMeasure,price,pricePerWeight,referenceURL,screenshot,createdAt,reference_item_id
0,1,Chicken 'N Rib BBQ Sauce,455,mL,None,0.007231,https://www.foodbasics.ca/aisles/pantry/condim...,./screenshots/Kraft BBQ Sauce_2025-01-26_22-04...,2025-01-27T03:04:36.799Z,1
1,2,Halal Chicken Flavoured Bouillon Cubes,80,g,None,0.016125,https://www.foodbasics.ca/aisles/pantry/herbs-...,./screenshots/Knorr Chicken Bouillon_2025-01-2...,2025-01-27T03:04:40.077Z,3
2,3,"Gummy Berries, Swedish Berries",315,g,None,0.012667,https://www.foodbasics.ca/aisles/snacks/sweet-...,./screenshots/Maynards Swedish Berries_2025-01...,2025-01-27T03:04:43.059Z,4
3,4,"Classic Flavour Chips, Party Size",415,g,None,0.012024,https://www.foodbasics.ca/aisles/snacks/salty-...,./screenshots/Lay's Classic Chips_2025-01-26_2...,2025-01-27T03:04:46.990Z,7
4,5,Chewy Caramel and Nut Granola Bars Coated With...,172,g,None,0.011570,https://www.foodbasics.ca/aisles/snacks/sweet-...,./screenshots/Hershey's Milk Chocolate Bar_202...,2025-01-27T03:21:26.220Z,8
...,...,...,...,...,...,...,...,...,...,...
40,41,No Calorie Granulated Sweetener,275.0,g,None,0.038145,https://www.foodbasics.ca/aisles/pantry/baking...,./screenshots/44_Granulated Sugar_2025-02-02_1...,2025-02-02T23:37:19.428Z,44
41,42,Blueberry Bagel,452.0,g,None,0.004403,https://www.foodbasics.ca/aisles/bread-bakery-...,./screenshots/45_Muffins_2025-02-02_18-37-50.png,2025-02-02T23:37:41.421Z,45
42,43,Chocolate Cupcakes,284.0,g,None,0.014049,https://www.foodbasics.ca/aisles/bread-bakery-...,./screenshots/52_Cupcakes_2025-02-02_18-39-57.png,2025-02-02T23:39:49.005Z,52
43,44,Cooked Rotisserie Chicken Legs,600.0,g,None,0.016650,https://www.foodbasics.ca/aisles/deli-prepared...,./screenshots/54_Maple Leaf Seasoned Chicken_2...,2025-02-02T23:40:33.658Z,54


In [41]:
# # Create a mapping of referenceUrl to most recent price
# url_to_price = {}
# for _, row in json_scrapes_df.iterrows():
#     url = row['referenceUrl']
#     # Only update if URL not seen or this price is from a more recent date
#     if url not in url_to_price or row['createdAt'] > url_to_price[url]['createdAt']:
#         url_to_price[url] = {
#             'price': row['price'],
#             'createdAt': row['createdAt']
#         }

# print(url_to_price)
# # Update prices in df_proofs based on referenceUrl
# for idx, row in df_proofs.iterrows():
#     if row['referenceURL'] in url_to_price:
#         df_proofs.at[idx, 'price'] = url_to_price[row['referenceURL']]['price']

# # Show updated df_proofs
# df_proofs


In [42]:
# Display rows where price is None
df_proofs[df_proofs['price'].isna()]

# For each proof with missing price, find matching scrape by referenceURL and copy over all fields
for idx, proof_row in df_proofs[df_proofs['price'].isna()].iterrows():
    matching_scrapes = json_scrapes_df[json_scrapes_df['referenceUrl'] == proof_row['referenceURL']]
    
    if not matching_scrapes.empty:
        # Get most recent scrape
        latest_scrape = matching_scrapes.sort_values('createdAt', ascending=False).iloc[0]
        
        # Update all fields from scrape
        df_proofs.at[idx, 'name'] = latest_scrape['name']
        df_proofs.at[idx, 'quantity'] = latest_scrape['quantity'] 
        df_proofs.at[idx, 'unitOfMeasure'] = latest_scrape['unitOfMeasure']
        df_proofs.at[idx, 'price'] = latest_scrape['price']
        df_proofs.at[idx, 'createdAt'] = latest_scrape['createdAt']

# Show updated proofs with missing prices
df_proofs[df_proofs['price'].isna()]




,id,name,quantity,unitOfMeasure,price,pricePerWeight,referenceURL,screenshot,createdAt,reference_item_id
19,20,"Chocolate Chips Chewy Bars, Value Pack, Chewy",960.0,g,None,0.014573,https://www.foodbasics.ca/aisles/snacks/sweet-...,./screenshots/6_Bulk Chia Seeds_2025-02-02_14-...,2025-02-02T19:18:32.571Z,6
20,21,Chow Mein Instant Noodles,454.0,g,None,0.003943,https://www.foodbasics.ca/aisles/pantry/pasta-...,./screenshots/22_Mr. Noodles Instant Noodles_2...,2025-02-02T19:22:44.526Z,22
21,22,One Minute Oatmeal,900.0,g,None,0.004433,https://www.foodbasics.ca/aisles/pantry/cereal...,./screenshots/23_Quaker Oats_2025-02-02_14-23-...,2025-02-02T19:23:10.178Z,23
22,23,Cornstarch Baby Powder,624.0,g,None,0.006394,https://www.foodbasics.ca/aisles/baby/needs/he...,./screenshots/11_Cornstarch_2025-02-02_14-30-3...,2025-02-02T19:30:30.756Z,11
23,24,"Garlic And Onion Pasta Sauce, Garden Select",600.0,mL,None,0.004150,https://www.foodbasics.ca/aisles/pantry/canned...,./screenshots/24_Catelli Pasta_2025-02-02_14-3...,2025-02-02T19:30:53.784Z,24
...,...,...,...,...,...,...,...,...,...,...
40,41,No Calorie Granulated Sweetener,275.0,g,None,0.038145,https://www.foodbasics.ca/aisles/pantry/baking...,./screenshots/44_Granulated Sugar_2025-02-02_1...,2025-02-02T23:37:19.428Z,44
41,42,Blueberry Bagel,452.0,g,None,0.004403,https://www.foodbasics.ca/aisles/bread-bakery-...,./screenshots/45_Muffins_2025-02-02_18-37-50.png,2025-02-02T23:37:41.421Z,45
42,43,Chocolate Cupcakes,284.0,g,None,0.014049,https://www.foodbasics.ca/aisles/bread-bakery-...,./screenshots/52_Cupcakes_2025-02-02_18-39-57.png,2025-02-02T23:39:49.005Z,52
43,44,Cooked Rotisserie Chicken Legs,600.0,g,None,0.016650,https://www.foodbasics.ca/aisles/deli-prepared...,./screenshots/54_Maple Leaf Seasoned Chicken_2...,2025-02-02T23:40:33.658Z,54


In [48]:
# Connect to SQLite database
import sqlite3
conn = sqlite3.connect('../receipts-app/prisma/dev.db')

# Get rows that have prices
proofs_with_prices = df_proofs[df_proofs['price'].notna()]

# Update each row in the database
for _, row in proofs_with_prices.iterrows():
    update_sql = """
    UPDATE ProductPriceProof 
    SET name = ?,
        quantity = ?,
        unitOfMeasure = ?,
        price = ?,
        pricePerWeight = ?,
        referenceURL = ?,
        screenshot = ?,
        createdAt = datetime(?),
        reference_item_id = ?
    WHERE id = ?
    """
    conn.execute(update_sql, (
        row['name'],
        row['quantity'],
        row['unitOfMeasure'], 
        row['price'],
        row['pricePerWeight'],
        row['referenceURL'],
        row['screenshot'],
        row['createdAt'].strftime('%Y-%m-%d %H:%M:%S'),
        row['reference_item_id'],
        row['id']
    ))

conn.commit()
conn.close()

print(f"Updated {len(proofs_with_prices)} rows in database")


Updated 20 rows in database
